# Bond Pricing and Interest-Rate Risk

## Objective

This notebook examines whether the yield-curve interpolation method affects
the valuation and interest-rate sensitivity of fixed-rate coupon bonds.

The same bond is priced using:

1. a curve constructed by interpolating zero rates;
2. a curve constructed by interpolating discount factors.

The resulting prices, DV01, duration and convexity are compared.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

In [2]:
market_data = pd.DataFrame(
    {
        "maturity": [0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0],
        "zero_rate": [0.0420, 0.0435, 0.0445, 0.0438, 0.0415, 0.0400, 0.0390],
    }
)

market_data["discount_factor"] = np.exp(
    -market_data["zero_rate"] * market_data["maturity"]
)

curve_maturities = np.linspace(
    market_data["maturity"].min(),
    market_data["maturity"].max(),
    500,
)

In [3]:
zero_rate_interpolator = interp1d(
    market_data["maturity"],
    market_data["zero_rate"],
    kind="linear",
)

discount_factor_interpolator = interp1d(
    market_data["maturity"],
    market_data["discount_factor"],
    kind="linear",
)

zero_rates_from_zero_interpolation = zero_rate_interpolator(
    curve_maturities
)

discount_factors_from_zero_interpolation = np.exp(
    -zero_rates_from_zero_interpolation * curve_maturities
)

discount_factors_from_discount_interpolation = (
    discount_factor_interpolator(curve_maturities)
)

zero_rates_from_discount_interpolation = (
    -np.log(discount_factors_from_discount_interpolation)
    / curve_maturities
)

## Bond cash flows

A fixed-rate bond is represented by its payment times and cash flows. Each cash
flow is discounted using the relevant term structure.

For payment times $t_i$, cash flows $CF_i$, and discount curve $P(0,t)$, the bond price is

$$
B_0=\sum_{i=1}^{n} CF_i P(0,t_i).
$$

In [4]:
face_value = 100.0
coupon_rate = 0.045
maturity = 7.0
payments_per_year = 2

coupon_payment = (
    face_value
    * coupon_rate
    / payments_per_year
)

payment_times = np.arange(
    1 / payments_per_year,
    maturity + 1 / payments_per_year,
    1 / payments_per_year,
)

cashflows = np.full(
    payment_times.shape,
    coupon_payment,
)

cashflows[-1] += face_value

bond_cashflows = pd.DataFrame(
    {
        "payment_time": payment_times,
        "cashflow": cashflows,
    }
)

bond_cashflows

,payment_time,cashflow
0,0.5,2.25
1,1.0,2.25
2,1.5,2.25
3,2.0,2.25
4,2.5,2.25
5,3.0,2.25
6,3.5,2.25
7,4.0,2.25
8,4.5,2.25
9,5.0,2.25


In [5]:
def price_bond(
    payment_times: np.ndarray,
    cashflows: np.ndarray,
    curve_times: np.ndarray,
    discount_factors: np.ndarray,
) -> float:
    """Return the present value of deterministic bond cash flows."""

    cashflow_discount_factors = np.interp(
        payment_times,
        curve_times,
        discount_factors,
    )

    return float(
        np.sum(cashflows * cashflow_discount_factors)
    )

In [6]:
price_from_zero_interpolation = price_bond(
    payment_times,
    cashflows,
    curve_maturities,
    discount_factors_from_zero_interpolation,
)

price_from_discount_interpolation = price_bond(
    payment_times,
    cashflows,
    curve_maturities,
    discount_factors_from_discount_interpolation,
)

price_difference = (
    price_from_discount_interpolation
    - price_from_zero_interpolation
)

print(
    "Price using zero-rate interpolation:",
    f"{price_from_zero_interpolation:.6f}",
)

print(
    "Price using discount-factor interpolation:",
    f"{price_from_discount_interpolation:.6f}",
)

print(
    "Price difference:",
    f"{price_difference:.6f}",
)

Price using zero-rate interpolation: 102.596486
Price using discount-factor interpolation: 102.614575
Price difference: 0.018089


In [7]:
bond_cashflows["discount_factor_zero_method"] = np.interp(
    payment_times,
    curve_maturities,
    discount_factors_from_zero_interpolation,
)

bond_cashflows["discount_factor_discount_method"] = np.interp(
    payment_times,
    curve_maturities,
    discount_factors_from_discount_interpolation,
)

bond_cashflows["present_value_zero_method"] = (
    bond_cashflows["cashflow"]
    * bond_cashflows["discount_factor_zero_method"]
)

bond_cashflows["present_value_discount_method"] = (
    bond_cashflows["cashflow"]
    * bond_cashflows["discount_factor_discount_method"]
)

bond_cashflows["present_value_difference"] = (
    bond_cashflows["present_value_discount_method"]
    - bond_cashflows["present_value_zero_method"]
)

bond_cashflows

,payment_time,cashflow,discount_factor_zero_method,discount_factor_discount_method,present_value_zero_method,present_value_discount_method,present_value_difference
0,0.5,2.25,0.979219,0.979219,2.203243,2.203243,0.000000
1,1.0,2.25,0.957440,0.957436,2.154239,2.154231,-0.000008
2,1.5,2.25,0.936131,0.936139,2.106294,2.106313,0.000018
3,2.0,2.25,0.914855,0.914860,2.058425,2.058435,0.000011
4,2.5,2.25,0.895498,0.895856,2.014871,2.015677,0.000805
5,3.0,2.25,0.876872,0.876891,1.972962,1.973005,0.000043
6,3.5,2.25,0.859601,0.860804,1.934102,1.936808,0.002706
7,4.0,2.25,0.843159,0.844740,1.897107,1.900665,0.003558
8,4.5,2.25,0.827507,0.828677,1.861891,1.864523,0.002631
9,5.0,2.25,0.812606,0.812630,1.828364,1.828417,0.000053


In [8]:
assert price_from_zero_interpolation > 0
assert price_from_discount_interpolation > 0

assert np.isclose(
    price_from_zero_interpolation,
    bond_cashflows["present_value_zero_method"].sum(),
)

assert np.isclose(
    price_from_discount_interpolation,
    bond_cashflows["present_value_discount_method"].sum(),
)

print("Bond-pricing validation checks passed.")

Bond-pricing validation checks passed.


## Initial pricing comparison

The bond is priced under both interpolated term structures using identical cash
flows and payment dates. Any price difference therefore arises solely from the
choice of interpolation method.

The cash-flow table identifies which payment dates contribute most to the
valuation difference. Later sections examine whether the same modelling choice
also affects DV01, duration and convexity.

## Cash-flow present values

Before calculating interest-rate risk measures, the discounted value of each
cash flow is computed. This provides the building blocks for duration,
convexity and DV01 calculations.

In [9]:
def discounted_cashflows(
    payment_times: np.ndarray,
    cashflows: np.ndarray,
    curve_times: np.ndarray,
    discount_factors: np.ndarray,
) -> pd.DataFrame:
    """
    Return the discounted value of each bond cash flow.
    """

    interpolated_discount_factors = np.interp(
        payment_times,
        curve_times,
        discount_factors,
    )

    present_values = (
        cashflows
        * interpolated_discount_factors
    )

    return pd.DataFrame(
        {
            "payment_time": payment_times,
            "cashflow": cashflows,
            "discount_factor": interpolated_discount_factors,
            "present_value": present_values,
        }
    )

In [10]:
bond_zero_curve = discounted_cashflows(
    payment_times,
    cashflows,
    curve_maturities,
    discount_factors_from_zero_interpolation,
)

bond_discount_curve = discounted_cashflows(
    payment_times,
    cashflows,
    curve_maturities,
    discount_factors_from_discount_interpolation,
)

In [11]:
comparison = pd.DataFrame(
    {
        "payment_time": payment_times,
        "cashflow": cashflows,
        "PV (zero interpolation)": bond_zero_curve["present_value"],
        "PV (discount interpolation)": bond_discount_curve["present_value"],
    }
)

comparison["Difference"] = (
    comparison["PV (discount interpolation)"]
    - comparison["PV (zero interpolation)"]
)

comparison

,payment_time,cashflow,PV (zero interpolation),PV (discount interpolation),Difference
0,0.5,2.25,2.203243,2.203243,0.000000
1,1.0,2.25,2.154239,2.154231,-0.000008
2,1.5,2.25,2.106294,2.106313,0.000018
3,2.0,2.25,2.058425,2.058435,0.000011
4,2.5,2.25,2.014871,2.015677,0.000805
5,3.0,2.25,1.972962,1.973005,0.000043
6,3.5,2.25,1.934102,1.936808,0.002706
7,4.0,2.25,1.897107,1.900665,0.003558
8,4.5,2.25,1.861891,1.864523,0.002631
9,5.0,2.25,1.828364,1.828417,0.000053


## Interest-rate duration

Bond prices respond to changes in interest rates. One of the most widely used
measures of this sensitivity is Macaulay duration, which represents the
cash-flow-weighted average time to receipt.

This section investigates whether the choice of yield-curve interpolation
affects the estimated duration of the same bond.

## Macaulay duration

Macaulay duration measures the weighted-average time at which the bond's cash
flows are received, where the weights are given by the present value of each
cash flow.

For bond price $B_0$, the Macaulay duration is 

$$
D_{\mathrm{Mac}}
=
\frac{\sum_{i=1}^{n} t_i\,CF_i\,P(0,t_i)}
{B_0}.
$$

This quantity depends on the discount curve used to value the bond and may
therefore differ under alternative interpolation methods.

In [12]:
def macaulay_duration(
    discounted_cashflows: pd.DataFrame,
) -> float:
    """
    Return the Macaulay duration of a fixed-rate bond.
    """

    bond_price = discounted_cashflows["present_value"].sum()

    return float(
        (
            discounted_cashflows["payment_time"]
            * discounted_cashflows["present_value"]
        ).sum()
        / bond_price
    )

In [13]:
duration_zero_curve = macaulay_duration(
    bond_zero_curve,
)

duration_discount_curve = macaulay_duration(
    bond_discount_curve,
)

print(
    "Macaulay duration (zero interpolation):",
    f"{duration_zero_curve:.6f}",
)

print(
    "Macaulay duration (discount interpolation):",
    f"{duration_discount_curve:.6f}",
)

print(
    "Difference:",
    f"{duration_discount_curve-duration_zero_curve:.8f}",
)

Macaulay duration (zero interpolation): 6.101595
Macaulay duration (discount interpolation): 6.101393
Difference: -0.00020181


## Interpretation

The two interpolation methods produce almost identical Macaulay durations:
approximately 6.102 years in both cases. The difference is only about
0.0002 years, even though the bond prices differ by approximately 0.018.

This suggests that, for this bond, the interpolation choice has a measurable
effect on valuation but very little effect on the present-value-weighted timing
of the cash flows. The next step is to examine price sensitivity directly
through curve shocks and DV01.

## Parallel curve shocks and DV01

DV01 measures the change in a bond's value
resulting from a one basis-point parallel shift in the yield curve.

Rather than relying on a closed-form approximation, the bond is repriced after
shifting the entire zero curve by ±1 basis point. This approach is consistent
with practical fixed-income risk management, where sensitivities are typically
obtained through repricing.

In [14]:
def shock_zero_rates(
    zero_rates: np.ndarray,
    shock_bp: float,
) -> np.ndarray:
    """
    Apply a parallel basis-point shock to a zero-rate curve.
    """

    return zero_rates + shock_bp / 10000

In [15]:
def zero_rates_to_discount_factors(
    zero_rates: np.ndarray,
    maturities: np.ndarray,
) -> np.ndarray:
    """
    Convert continuously compounded zero rates to discount factors.
    """

    return np.exp(
        -zero_rates * maturities
    )

In [16]:
zero_up = shock_zero_rates(
    zero_rates_from_zero_interpolation,
    1,
)

zero_down = shock_zero_rates(
    zero_rates_from_zero_interpolation,
    -1,
)

discount_up = shock_zero_rates(
    zero_rates_from_discount_interpolation,
    1,
)

discount_down = shock_zero_rates(
    zero_rates_from_discount_interpolation,
    -1,
)

In [17]:
discount_zero_up = zero_rates_to_discount_factors(
    zero_up,
    curve_maturities,
)

discount_zero_down = zero_rates_to_discount_factors(
    zero_down,
    curve_maturities,
)

discount_discount_up = zero_rates_to_discount_factors(
    discount_up,
    curve_maturities,
)

discount_discount_down = zero_rates_to_discount_factors(
    discount_down,
    curve_maturities,
)

## Repricing under shocked curves

Each shocked yield curve is converted into discount factors and the bond is
repriced. These prices are subsequently used to estimate DV01 and convexity
using finite-difference approximations.

In [18]:
price_zero_up = price_bond(
    payment_times,
    cashflows,
    curve_maturities,
    discount_zero_up,
)

price_zero_down = price_bond(
    payment_times,
    cashflows,
    curve_maturities,
    discount_zero_down,
)

price_discount_up = price_bond(
    payment_times,
    cashflows,
    curve_maturities,
    discount_discount_up,
)

price_discount_down = price_bond(
    payment_times,
    cashflows,
    curve_maturities,
    discount_discount_down,
)

In [19]:
print("Zero-rate interpolation")
print(f"Price (+1 bp): {price_zero_up:.6f}")
print(f"Price (-1 bp): {price_zero_down:.6f}")

print()

print("Discount-factor interpolation")
print(f"Price (+1 bp): {price_discount_up:.6f}")
print(f"Price (-1 bp): {price_discount_down:.6f}")

Zero-rate interpolation
Price (+1 bp): 102.533906
Price (-1 bp): 102.659107

Discount-factor interpolation
Price (+1 bp): 102.551986
Price (-1 bp): 102.677204


## DV01

DV01 measures the change in a bond's value
resulting from a one basis-point parallel shift in the yield curve.

The measure is estimated using a central finite-difference approximation,

$$
\mathrm{DV01}
=
\frac{P_{-1\text{bp}}-P_{+1\text{bp}}}{2},
$$

where $P_{-1\mathrm{bp}}$ and $P_{+1\mathrm{bp}}$ are the bond prices after parallel downward and upward shifts of one basis point, respectively.

In [20]:
def dv01(
    price_up: float,
    price_down: float,
) -> float:
    """
    Estimate DV01 using a central finite-difference approximation.
    """

    return (price_down - price_up) / 2

In [21]:
dv01_zero = dv01(
    price_zero_up,
    price_zero_down,
)

dv01_discount = dv01(
    price_discount_up,
    price_discount_down,
)

print(
    f"DV01 (zero-rate interpolation): {dv01_zero:.8f}"
)

print(
    f"DV01 (discount-factor interpolation): {dv01_discount:.8f}"
)

print(
    "Difference:",
    f"{dv01_discount-dv01_zero:.10f}",
)

DV01 (zero-rate interpolation): 0.06260019
DV01 (discount-factor interpolation): 0.06260916
Difference: 0.0000089667


## Interpretation

The two interpolation methods produce almost identical DV01 estimates, at
approximately 0.0626 per 100 of face value. The difference between the two estimates is less than $10^{-5}$.

For this bond, the interpolation choice therefore has a more visible effect on
the level of the valuation than on its sensitivity to a small parallel shift
in the yield curve. This suggests that the local differences between the two
term structures have limited impact on first-order parallel-rate risk in this
example.

## Convexity

Duration provides a first-order approximation to the relationship between bond
prices and interest rates. Convexity captures the curvature of this
relationship and improves the approximation for larger yield changes.

Using the same shocked prices, convexity is estimated by the central
finite-difference approximation

$$
C
=
\frac{P_{-1\text{bp}}
-
2P_0
+
P_{+1\text{bp}}}
{P_0(\Delta y)^2},
$$

where $P_0$ is the original bond price and $\Delta y = 0.0001$.

In [22]:
def convexity(
    price_down: float,
    price: float,
    price_up: float,
    shock: float = 0.0001,
) -> float:
    """
    Estimate convexity using a central finite-difference approximation.
    """

    return (
        price_down
        - 2 * price
        + price_up
    ) / (price * shock**2)

In [23]:
convexity_zero = convexity(
    price_zero_down,
    price_from_zero_interpolation,
    price_zero_up,
)

convexity_discount = convexity(
    price_discount_down,
    price_from_discount_interpolation,
    price_discount_up,
)

print(
    f"Convexity (zero-rate interpolation): {convexity_zero:.6f}"
)

print(
    f"Convexity (discount-factor interpolation): {convexity_discount:.6f}"
)

print(
    "Difference:",
    f"{convexity_discount-convexity_zero:.8f}",
)

Convexity (zero-rate interpolation): 40.556365
Convexity (discount-factor interpolation): 40.553850
Difference: -0.00251519


## Interpretation

The convexity estimates obtained from the two interpolated curves are very
similar. This suggests that, for this bond, the interpolation choice has only
a limited effect on second-order interest-rate sensitivity.

Combined with the duration and DV01 results, this indicates that the principal
effect of the interpolation method is on the level of the bond valuation,
rather than on its first- or second-order sensitivity to small parallel
movements in the yield curve.

# Findings

This notebook investigated whether the method used to interpolate the yield
curve materially affects bond valuation and interest-rate risk.

For the bond considered here, the interpolation choice produced a small but
measurable difference in the bond price. In contrast, Macaulay duration, DV01
and convexity were almost identical under the two term structures.

These results suggest that, for this example, the interpolation method has a
greater influence on the valuation itself than on first- or second-order
parallel-rate risk measures.

This analysis provides a foundation for the next stage of the project, where
the same interpolation methods will be applied to interest-rate swaps, whose
valuation depends more directly on the shape of the underlying term structure.